In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_date

def SCD2Function(catalog_name, db_name, table_name_stage, catalog_source,
                 db_name_source, table_name_source, column_names):
    # Start Spark session if not already started
    spark = SparkSession.builder.appName("SCD2 Function").getOrCreate()

    # Use the specified catalog and database
    spark.sql(f"use catalog {catalog_name}")
    spark.sql(f"use schema {db_name}")

    # Drop the staging table if it exists and create a new one
    spark.sql(f"DROP TABLE IF EXISTS {table_name_stage};")
    column_definitions = ", ".join([f"{col} STRING" for col in column_names])
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table_name_stage} ({column_definitions}) USING DELTA;")

    # Truncate the staging table
    spark.sql(f"TRUNCATE TABLE {table_name_stage};")

    # Insert data into the staging table
    insert_query = f"INSERT INTO {table_name_stage} SELECT " + ", ".join([f"TRY_CAST({col} AS STRING)" for col in column_names]) + f" FROM {catalog_source}.{db_name_source}.{table_name_source};"
    spark.sql(insert_query)

    

# Example usage:
column_names = ["id", "body_html", "title", "handle", "product_type", "vendor", "created_at", "status"]
SCD2Function("streaming1", "silver", "stg_product", "hive_metastore","_fivetran_setup_test", "product", column_names)


In [ ]:
SCD2function() 

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,2,0,0


In [ ]:
df=spark.sql(f"select * from {table_name_dim};")
display(df)